# Où construire des centres de distribution ?
 - Correction
Considérons un ensemble de $m$ emplacements $\{1,\dots, m\}$ dans lesquels un centre de distribution peut être ouvert.
En outre, considérons un ensemble de $n$ clients $\{1,\dots, n\}$ qui doivent être servis par un centre de distribution (ouvert). La distance entre le $i^\text{ième}$ client et le $j^\text{ième}$ centre de distribution
est $d_{ij}$ ($d_{ij}\ge 0$). Le nombre maximal de centres de distribution pouvant être ouverts est $q$.

## Question 1

Écrivez un modèle de PLNE pour minimiser la distance totale parcourue par les clients. Identifiez les
décisions qui doivent être prises ainsi que les variables de décision correspondantes qui doivent être
utilisées. Identifiez et commentez la fonction objectif du problème et les contraintes.


**CORRECTION :**


Soit $y_i = \left\{ \begin{array}{ll} 1 & \text{si le centre de distribution numéro $i$ est ouvert}\\ 0 & \text{sinon}\end{array} \right .~~$ pour tout $i \in \{1,\dots, m\}$.

Soit $x_{ij} = \left\{ \begin{array}{ll} 1 & \text{si le client numéro $j$ est servi par le centre de distribution numéro $i$}\\ 0 & \text{sinon}\end{array} \right .~~$ pour tout $i \in \{1,\dots, m\}$ et $j \in \{1,\dots, n\}$.



Le problème peut être formulé comme suit.
$$
\begin{align}
&\min \sum_{i = 1}^m \sum_{j = 1}^n d_{ij}x_{ij}\\
&\sum_{i=1}^n y_{i}  \le q\\
&\sum_{i=1}^n x_{ij}  = 1 & \quad & \forall j=1,\dots,n\\
& x_{ij} \le y_i & & \forall i=1,\dots,m, \forall j=1,\dots,n\\
& y_i \in \{0, 1\} & & \forall i=1,\dots,m\\
& x_{ij} \ge 0 & & \forall i=1,\dots,m, \forall j=1,\dots,n
\end{align}
$$

## Question 3

Considérons maintenant le cas suivant dans lequel il y a 4 emplacements potentiels ($m = 4$) et 4
clients ($n = 4$), et un nombre maximal de 2 centres de distribution pouvant être ouverts ($q = 2$). Les
distances entre les clients et les centres de distribution sont :

| &nbsp;          | Client 1 | Client 2 | Client 3 | Client 4 |
|-----------------|---:|---:|---:|---:|
| Centre de distribution 1 | 0 | 10 | 30 | 40 |
| Centre de distribution 2 | 10 | 0 | 5 | 25 |
| Centre de distribution 3 | 30 | 5 | 0 | 50 |
| Centre de distribution 4 | 40 | 25 | 50 | 0 |



Écrivez le PLNE et implémentez-le pour cette instance spécifique.


In [ ]:
q = 2
distances = [
    [ 0, 10, 30, 40],
    [10,  0,  5, 25],
    [30,  5,  0, 50],
    [40, 25, 50,  0]
]

In [ ]:
%pip install mip

In [ ]:
from mip import *

**CORRECTION :**


PLNE pour l'instance spécifique

$$
\begin{align}
& \min 10 x_{12} + 30 x_{13} + 40 x_{14} + 10 x_{21} + 5 x_{23} + 25 x_{24} + 30 x_{31} + 5 x_{32} + 50 x_{33} + 40 x_{41} + 25 x_{42} + 50 x_{43}\\
& y_1 + y_2 + y_3 + y_4 \le 2\\
&x_{11} + x_{12} + x_{13} + x_{14} = 1\\
&x_{21} + x_{22} + x_{23} + x_{24} = 1\\
&x_{31} + x_{32} + x_{33} + x_{34} = 1\\
&x_{41} + x_{42} + x_{43} + x_{44} = 1\\
&x_{11} \le y_1, x_{12} \le y_1, x_{13} \le y_1, x_{14} \le y_1,\\
&x_{21} \le y_1, x_{22} \le y_1, x_{23} \le y_1, x_{24} \le y_2,\\
&x_{31} \le y_1, x_{32} \le y_1, x_{33} \le y_1, x_{34} \le y_3,\\
&x_{41} \le y_1, x_{42} \le y_1, x_{43} \le y_1, x_{44} \le y_4,\\
&x \geq \mathbf{0}\\
&y_1,\dots,y_4 \in \{0,,1\}
\end{align}
$$

In [ ]:
##################
#   Correction   #
##################


def distribution_centers(q, distances):
    m = Model()
    I = range(len(distances))
    J = range(len(distances[0]))
    x = [
            [m.add_var() for j in J] for i in I
    ]
    y = [ m.add_var(var_type = BINARY) for i in I]
    m.objective = minimize(xsum(distances[i][j] * x[i][j] for i in I for j in J))

    # Open q distribution centers
    m += xsum(y[i] for i in I) <= q

    # for each client
    for j in J:
        m += xsum(x[i][j] for i in I) == 1

    # for each distribution center and each client
    for i in I:
        for j in J:
            m += x[i][j] <= y[i]
            
    m.verbose = 0 # to remove the output of the solution process
    status = m.optimize()
    print(f"Total distance: {round(m.objective_value)}")
    for i in I:
        if round(y[i].x) == 1:
            clients = [j+1 for j in J if round(x[i][j].x) == 1]
            print(f"Distribution center {i+1} is opened. Clients {clients} are served from it")

distribution_centers(q, distances)      